In [5]:
# échauffement : construire une URL

def url(num=3,typ="rue", nom="tourelles"):
    return f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1"

print(url())

https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles,Paris&limit=1


In [6]:
# échauffement : sélectionner des lignes avec iloc et loc 

import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

# 1. sub contient : "number" : [18,12], "name":["BERNARDINS","Rivoli"]
# index : 1, 3

# 2. 
# df.iloc[:2] = pd.DataFrame({
#       "number": [3,18]
#       "name": ["tourelles","BERNARDINS"]})

# df.loc[:2] = pd.DataFrame({
#       "number": [3,18,7]
#       "name": ["tourelles","BERNARDINS","Bouvart"]})

# iloc exclu l'index 2, loc l'inclu

# 3. 
# sub.iloc[:2] = pd.DataFrame({
#       "number": [18,12]
#       "name": ["BERNARDINS","Rivoli"]})
# 
# sub.loc[:2] = pd.DataFrame({
#       "number": [18]
#       "name": ["BERNARDINS"]})
# iloc compte les positions là où loc compte les indices

# 4. sub.loc[0] renvoie une erreur car il n'y a pas d'index 0
# sub.iloc[0] renvoie la première ligne de sub

# 5.

print(sub["name"].iloc[0])
print(sub["name"].loc[sub.index[0]])

BERNARDINS
BERNARDINS


In [7]:
# géolocalisation


addresses = pd.read_csv('data/addresses.csv')
addresses.head(4)

addresses_small=addresses.loc[:20]
addresses_small.to_csv("addresses-small.csv")


In [8]:
# une par une 

# on obtient un fichier json
# on obtient le même fichier mais seulement avec l'index 0


import requests

def localize_one(num, typ, nom):
    url_=url(num,typ,nom)
    r=requests.get(url_, auth=('user', 'pass'))
    if r.status_code < 200 or r.status_code >=300:
        return None
    return r.json()


import time 

t1=time.time()
print(localize_one(18, 'rue', 'BERNARDINS'))
t2=time.time()

print("20000 requests prendraient", 20000*(t2-t1),"secondes")

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]}, 'properties': {'label': '18 Rue des Bernardins 75005 Paris', 'score': 0.7544495104895104, 'housenumber': '18', 'id': '75105_0896_00018', 'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02', 'name': '18 Rue des Bernardins', 'postcode': '75005', 'citycode': '75105', 'x': 652355.3, 'y': 6861340.65, 'city': 'Paris', 'district': 'Paris 5e Arrondissement', 'context': '75, Paris, Île-de-France', 'type': 'housenumber', 'importance': 0.68356, 'depcode': '75', 'street': 'Rue des Bernardins', '_type': 'address'}}], 'query': '18 rue BERNARDINS,Paris'}
20000 requests prendraient 8008.39900970459 secondes


In [ ]:
# en un seul coup


import requests
import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    # we don't care about the index for now
    # but note that if we want to merge the result
    # with the initial data, there is a need 
    # for a unique index...

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    # this time the output is csv, not json
    # note that read_csv requires a file-like object
    # and we have a str, so io.StringIO comes to the rescue
    result = pd.read_csv(io.StringIO(response.text))

    return result


# votre code

def localize_many(filename, col_number, col_type, col_name, col_city):
    df = pd.read_csv(filename, encoding='utf-8')
    geoloc = mass_post(filename,col_number,col_type,col_name,col_city)
    geoloc = geoloc[['latitude','longitude','result_city','result_type']]
    return df.merge(geoloc,left_index=True,right_index=True)

# try your code on the small sample for starters

addresses_small = localize_many("data/addresses-small.csv", "number", "type", "name", "city")
print(addresses_small)

# Pour enlever les adresses non parisiennes

addresses_small = addresses_small.fillna(0)
addresses_small=addresses_small.loc[addresses_small['result_city']=='Paris']
print(addresses_small)

# On enregistre le fichier  

# addresses=localize_many("data/addresses.csv","number","type","name","city")
# addresses.to_csv(index=False)

# Le code ne se compile pas correctement


   number       type                name   city   latitude  longitude  \
0       5        rue              RIQUET  Paris  48.888134   2.376448   
1     141  boulevard              ITALIE  Paris  46.669103  -1.418872   
2     234        rue  FAUBOURG ST-HONORÉ  Paris  48.876160   2.302766   
3      62        rue        MICHEL BIZOT  Paris  48.835423   2.402040   
4       6        rue                OURS  Paris  48.863311   2.351775   
5      30        rue          PETIT-MUSC  Paris  48.852945   2.364722   
6       3    impasse            MASSONET  Paris  48.895498   2.351580   
7      56        rue            PANOYAUX  Paris  48.867023   2.388347   
8      16        rue             THÉRÈSE  Paris  48.866374   2.335212   
9      11        rue         JEAN ROBERT  Paris  48.889520   2.358170   

        result_city  result_type  
0             Paris  housenumber  
1  La Roche-sur-Yon       street  
2             Paris  housenumber  
3             Paris  housenumber  
4             Paris  

'\naddresses=localize_many("data/addresses.csv","number","type","name","city")\naddresses.to_csv(index=False)\n'

In [15]:
import folium

# ~ chatelet

CENTER = 48.856542, 2.347614

# pour commencer on va recharger la dataframe précédente
import pandas as pd

addresses = pd.read_csv("data/addresses-geoloc.csv")

# et en faire un petit échantillon

addresses_small = addresses.iloc[:20]

# créez une map centrée sur ce point et de zoom 13
# n'oubliez pas de l'afficher,
# et vérifiez que vous voyez bien Paris, 
# que vous pouvez zoomer et vous déplacer, ...

# votre code
def paris_map():
    m = folium.Map(location=CENTER, zoom_start=12)
    return m

# pour l'afficher
paris_map()


In [ ]:
# your code here

def map_addresses(geoloc):
    """
    creates folium map centered on Paris, with the various addresses
    contained in the input dataframe shown as a marker
    """
    for _, row in geoloc.iterrows():
        folium.Marker(location=(row['latitude'],row['longitude']),tooltip="Clique").add_to(m)
        print(row)

map_addresses(addresses_small)
paris_map()
    


number                   5
type                   rue
name                RIQUET
city                 Paris
latitude         48.888134
longitude         2.376448
result_city          Paris
result_type    housenumber
Name: 0, dtype: object
number                 141
type             boulevard
name                ITALIE
city                 Paris
latitude         48.820646
longitude         2.359275
result_city          Paris
result_type    housenumber
Name: 1, dtype: object
number                        234
type                          rue
name           FAUBOURG ST-HONORÉ
city                        Paris
latitude                 48.87616
longitude                2.302766
result_city                 Paris
result_type           housenumber
Name: 2, dtype: object
number                   62
type                    rue
name           MICHEL BIZOT
city                  Paris
latitude          48.835423
longitude           2.40204
result_city           Paris
result_type     housenumber
Nam